# 00 · Обзор публичных задач

Разогрев (R-L0-1): быстро посмотреть на обе публичные задачи — размеры, типы признаков, пропуски, таргет.
Код **универсальный**: таргет и id берём из `sample_submission.csv`, а не прописываем руками — так же поступает агент.
Выводы, полезные агенту, переносим в `docs/experiments.md` / `playbook.yaml`, а не оставляем здесь.

Перед коммитом: `Clear All Outputs`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Ноутбук лежит в research/notebooks/ — поднимаемся к корню репозитория
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
TASKS = sorted(p.parent for p in (ROOT / "tasks" / "public").glob("*/task.md"))
[t.name for t in TASKS]

In [ ]:
def load_task(task_dir: Path) -> dict:
    """Читает train/test/sample_submission и определяет id и таргет по шаблону ответа."""
    train = pd.read_csv(task_dir / "train.csv")
    test = pd.read_csv(task_dir / "test.csv")
    sample = pd.read_csv(task_dir / "sample_submission.csv", nrows=5)
    id_col, target = sample.columns[0], sample.columns[1]
    features = [c for c in train.columns if c not in (id_col, target)]
    return {"name": task_dir.name, "train": train, "test": test, "id": id_col, "target": target, "features": features}


tasks = {t.name: load_task(t) for t in TASKS}

## 1. Паспорт задач

In [ ]:
def passport(t: dict) -> dict:
    tr, te, feats = t["train"], t["test"], t["features"]
    num = tr[feats].select_dtypes("number").columns
    return {
        "train": tr.shape,
        "test": te.shape,
        "id": t["id"],
        "target": t["target"],
        "target dtype": str(tr[t["target"]].dtype),
        "числовых": len(num),
        "категориальных": len(feats) - len(num),
        "колонок с пропусками": int((tr[feats].isna().mean() > 0).sum()),
        "дубликатов строк": int(tr[feats].duplicated().sum()),
        "RAM train, МБ": round(tr.memory_usage(deep=True).sum() / 2**20, 1),
    }


pd.DataFrame({name: passport(t) for name, t in tasks.items()})

## 2. Пропуски (топ-10 колонок)

In [ ]:
for name, t in tasks.items():
    na = t["train"][t["features"]].isna().mean().sort_values(ascending=False)
    print(f"== {name}")
    na = na[na > 0]
    print(na.head(10).round(3).to_string() if len(na) else "пропусков нет")

## 3. Категориальные признаки: кардинальность и новые значения в test

In [ ]:
for name, t in tasks.items():
    tr, te = t["train"], t["test"]
    cats = tr[t["features"]].select_dtypes(exclude="number").columns
    rows = [
        {"колонка": c, "уникальных": tr[c].nunique(), "новых в test": len(set(te[c].dropna()) - set(tr[c].dropna()))}
        for c in cats
    ]
    print(f"== {name}")
    print(pd.DataFrame(rows).to_string(index=False) if rows else "категориальных нет")

## 4. Таргет

Регрессия — распределение и асимметрия (нужен ли log-таргет?). Классификация — доли классов (дисбаланс → Macro F1, стратификация).

In [ ]:
fig, axes = plt.subplots(1, len(tasks), figsize=(6 * len(tasks), 4))
for ax, (name, t) in zip(axes, tasks.items(), strict=True):
    y = t["train"][t["target"]]
    if pd.api.types.is_numeric_dtype(y) and y.nunique() > 20:
        y.plot.hist(bins=60, ax=ax)
        ax.set_title(f"{name}: {t['target']} (skew={y.skew():.2f})")
    else:
        y.value_counts().plot.barh(ax=ax, logx=True)
        ax.set_title(f"{name}: {t['target']} (лог. шкала)")
plt.tight_layout()

In [ ]:
for name, t in tasks.items():
    y = t["train"][t["target"]]
    if not (pd.api.types.is_numeric_dtype(y) and y.nunique() > 20):
        share = y.value_counts(normalize=True)
        print(f"== {name}: самый частый / самый редкий класс = {share.iloc[0] / share.iloc[-1]:.0f}x")
        print(share.round(4).to_string())

## 5. Сдвиг train → test по числовым признакам

Грубая проверка: насколько средние в test отличаются от train (в стандартных отклонениях train).
Большой сдвиг — повод думать о схеме валидации (например, по времени или группам).

In [ ]:
for name, t in tasks.items():
    tr, te = t["train"], t["test"]
    num = tr[t["features"]].select_dtypes("number").columns
    shift = ((te[num].mean() - tr[num].mean()) / tr[num].std().replace(0, 1)).abs().sort_values(ascending=False)
    print(f"== {name}")
    print(shift.head(8).round(3).to_string())

## Выводы первого прогона (21.10)

Цифры — из прогона этого ноутбука; проверять гипотезы в R-L1-1.

| Наблюдение | Где | Универсальная мысль для агента |
|---|---|---|
| `Year_Factor` в test сдвинут на ~1.9σ относительно train | regression | test может быть «из будущего» → случайный KFold переоценивает качество. Проверять сдвиг train→test, особенно по колонкам времени/года (⚠️ гипотеза: валидация по времени) |
| ~38% строк train — дубликаты по признакам (33 442 из 87 670) | multiclass | дубликаты попадают и в train, и в valid фолда → оптимистичная CV. Предупреждение в `DataProfile` (⚠️ гипотеза: группировать дубликаты в одном фолде) |
| В test есть значения категорий, которых нет в train (`state`: 2) | multiclass | кодирование категорий обязано переживать новые значения |
| Самый частый класс в 431 раз чаще самого редкого | multiclass | стратифицированная CV, веса классов, подбор порогов под Macro F1 |
| Пропуски до 62% в колонке | regression | бустинги терпят NaN; признак «пропуск есть» — кандидат в playbook |

Специфично только для этих задач (на private не пригодится): конкретные колонки погоды и протоколов.
